# 01 — Dataset Audit and Anomaly Visualisation

**Author:** Sherif Abd El-Rady

**Objective:** Load the MVTec AD dataset (or synthetic fallback), analyse the class distribution, visualise normal vs defect images side-by-side, and inspect ground-truth masks.

This notebook establishes what we are detecting, which categories of defects exist, and what the data looks like before any model touches it.


In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

ROOT     = Path('..').resolve()
MVTEC    = ROOT / 'data' / 'mvtec'
CATEGORY = 'bottle'
USE_SYNTHETIC = not (MVTEC / CATEGORY / 'train' / 'good').exists()

print(f'Category: {CATEGORY}')
print(f'Mode: {"SYNTHETIC (MVTec not found)" if USE_SYNTHETIC else "MVTec AD"}')

In [ ]:
# ── 1.0  Generate synthetic data if MVTec not available ───────────────────────
if USE_SYNTHETIC:
    print('Generating synthetic normal + defect images for pipeline validation...')
    rng = np.random.default_rng(42)

    synth_root = ROOT / 'data' / 'synthetic'
    for split in ['train/good', 'test/good', 'test/scratch', 'ground_truth/scratch']:
        (synth_root / split).mkdir(parents=True, exist_ok=True)

    def make_surface(rng, w=256, h=256, defect=False):
        """Generate a synthetic industrial surface (bottle-like)."""
        # Base: gradient + noise — simulates uniform product surface
        base = np.zeros((h, w, 3), dtype=np.float32)
        base[:, :, 0] = np.linspace(80, 120, w)  # horizontal gradient
        base[:, :, 1] = 90
        base[:, :, 2] = 100
        noise = rng.normal(0, 6, (h, w, 3))
        img = np.clip(base + noise, 0, 255).astype(np.uint8)

        mask = np.zeros((h, w), dtype=np.uint8)
        if defect:
            # Random scratch
            x1, y1 = rng.integers(30, w-30, size=2)
            x2, y2 = rng.integers(30, w-30, size=2)
            cv2.line(img,   (x1, y1), (x2, y2), (200, 200, 50), 3)
            cv2.line(mask,  (x1, y1), (x2, y2), 255, 5)

        return img, mask

    # Training normal images
    for i in range(60):
        img, _ = make_surface(rng, defect=False)
        cv2.imwrite(str(synth_root / 'train' / 'good' / f'{i:03d}.png'), img)
    # Test normal
    for i in range(15):
        img, _ = make_surface(rng, defect=False)
        cv2.imwrite(str(synth_root / 'test' / 'good' / f'{i:03d}.png'), img)
    # Test defect + masks
    for i in range(20):
        img, mask = make_surface(rng, defect=True)
        cv2.imwrite(str(synth_root / 'test' / 'scratch' / f'{i:03d}.png'), img)
        cv2.imwrite(str(synth_root / 'ground_truth' / 'scratch' / f'{i:03d}_mask.png'), mask)

    MVTEC    = synth_root
    CATEGORY = ''
    print(f'✓ Synthetic dataset: 60 train / 15 test normal / 20 test defect')
    print('  Note: results on synthetic data only validate the pipeline, not the model.')

In [ ]:
# ── 1.1  Inventory the dataset ────────────────────────────────────────────────
base = MVTEC / CATEGORY if CATEGORY else MVTEC
train_dir = base / 'train' / 'good'
test_dir  = base / 'test'
gt_dir    = base / 'ground_truth'

train_paths = sorted(train_dir.glob('*.png')) + sorted(train_dir.glob('*.jpg'))

# Test split — iterate subdirectories
test_by_class = defaultdict(list)
for cls_dir in sorted(test_dir.iterdir()):
    if cls_dir.is_dir():
        imgs = sorted(cls_dir.glob('*.png')) + sorted(cls_dir.glob('*.jpg'))
        test_by_class[cls_dir.name] = imgs

n_train     = len(train_paths)
n_test      = sum(len(v) for v in test_by_class.values())
n_normal_t  = len(test_by_class.get('good', []))
n_defect_t  = n_test - n_normal_t
defect_classes = [k for k in test_by_class if k != 'good']

print(f'=== Dataset Inventory: {CATEGORY or "synthetic"} ===')
print(f'Train (normal only) : {n_train}')
print(f'Test total          : {n_test}  ({n_normal_t} normal  |  {n_defect_t} defect)')
print(f'Defect classes      : {defect_classes}')
print(f'\nPer-class test counts:')
for cls, imgs in sorted(test_by_class.items()):
    tag = '(normal)' if cls == 'good' else '(defect)'
    print(f'  {cls:<25} {len(imgs):>4} images  {tag}')

In [ ]:
# ── 1.2  Class distribution bar chart ────────────────────────────────────────
classes = sorted(test_by_class.keys())
counts  = [len(test_by_class[c]) for c in classes]
colors  = ['#2ecc71' if c == 'good' else '#e74c3c' for c in classes]

fig, ax = plt.subplots(figsize=(max(8, len(classes) * 1.2), 4))
bars = ax.bar(classes, counts, color=colors, edgecolor='white', linewidth=0.8)
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_ylabel('Number of images')
ax.set_title(f'Test set class distribution — {CATEGORY or "synthetic"}\nGreen = normal  |  Red = defect')
ax.set_xticklabels(classes, rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

ratio = n_defect_t / n_test
print(f'Defect rate in test set: {ratio:.1%}')
print('Note: training set is 100% normal — this is unsupervised anomaly detection.')

In [ ]:
# ── 1.3  Visualise normal vs defect + ground-truth mask ──────────────────────
n_show = min(3, len(defect_classes))
fig    = plt.figure(figsize=(15, n_show * 3.5))
gs     = gridspec.GridSpec(n_show, 4, figure=fig, hspace=0.4, wspace=0.1)

# Show one normal image
normal_path = train_paths[0]
normal_img  = cv2.cvtColor(cv2.imread(str(normal_path)), cv2.COLOR_BGR2RGB)

for row, cls in enumerate(defect_classes[:n_show]):
    defect_paths = test_by_class[cls]
    defect_path  = defect_paths[0]
    defect_img   = cv2.cvtColor(cv2.imread(str(defect_path)), cv2.COLOR_BGR2RGB)

    # Ground truth mask
    mask_path = gt_dir / cls / (defect_path.stem + '_mask.png')
    if not mask_path.exists():
        mask_path = gt_dir / cls / (defect_path.stem + '.png')
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) if mask_path.exists() else None

    # Difference
    h, w = defect_img.shape[:2]
    norm_r = cv2.resize(normal_img, (w, h))
    diff = np.abs(defect_img.astype(int) - norm_r.astype(int)).mean(axis=2).astype(np.uint8)

    titles_imgs = [
        ('Normal (train)', normal_img, 'viridis'),
        (f'Defect: {cls}', defect_img, 'viridis'),
        ('GT Mask', mask, 'gray'),
        ('|Defect - Normal|', diff, 'hot'),
    ]
    for col, (title, im, cmap) in enumerate(titles_imgs):
        ax = fig.add_subplot(gs[row, col])
        if im is not None:
            ax.imshow(im, cmap=cmap if im.ndim == 2 else None)
        else:
            ax.text(0.5, 0.5, 'Mask\nnot found', ha='center', va='center',
                    transform=ax.transAxes, fontsize=8)
        ax.set_title(title, fontsize=8)
        ax.axis('off')

plt.suptitle('Normal vs Defect — Ground Truth Masks', fontsize=12, y=1.01)
plt.show()

## Summary

| Observation | Implication |
|---|---|
| Training = 100% normal | Model sees NO defects during training |
| Multiple defect types in test | Generalisation to unseen defect categories required |
| GT masks available | Pixel-level evaluation possible |
| Defect pixels ≪ normal pixels | Standard accuracy is misleading — use ROC-AUC |

## Next step

Open `02_classical_baseline.ipynb` to implement and evaluate the SSIM baseline.
